# B4 · 训练循环

前面所有零件，到这里组装起来。

**学完这一节你应该能**
- 背出训练循环的五个步骤
- 用 `Dataset` / `DataLoader` 组织数据，知道 batch / step / epoch 的关系
- 分清训练集和验证集，看懂两条 loss 曲线
- 说出 loss 不降时该按什么顺序查

In [ ]:
import torch


def check(name, ok, hint=""):
    if ok:
        print(f"✓ {name}")
    else:
        print(f"✗ {name}")
        if hint:
            print(f"   提示：{hint}")

## 1. 五个步骤

这是整门课最重要的一段，之后所有训练代码都是它的变体。

```python
pred = model(x)          # 1. 前向：算预测
loss = loss_fn(pred, y)  # 2. 算损失
optimizer.zero_grad()    # 3. 清空旧梯度
loss.backward()          # 4. 反向：算新梯度
optimizer.step()         # 5. 按梯度更新参数
```

**顺序能换吗？** `zero_grad` 必须在 `backward` 之前；`step` 必须在 `backward` 之后。其余相对自由，但按这个顺序写最不容易错。

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# 造一个回归数据集：y = 2x + 1 + 噪声
N = 500
X = torch.randn(N, 1) * 2
y = 2 * X + 1 + torch.randn(N, 1) * 0.1

print("X:", tuple(X.shape), " y:", tuple(y.shape))

In [ ]:
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

for step in range(200):
    pred = model(X)                    # 1
    loss = loss_fn(pred, y)            # 2
    optimizer.zero_grad()              # 3
    loss.backward()                    # 4
    optimizer.step()                   # 5

    if step % 50 == 0:
        print(f"step {step:>3} | loss {loss.item():.4f}")

print("\n学到的参数：", model.weight.item(), model.bias.item(), "（真实值 2 和 1）")

## 2. Dataset 与 DataLoader

数据只有 500 行时可以一把全喂进去。真实场景是几百万行，必须**分小批**（mini-batch）喂。

`DataLoader` 负责：切 batch、打乱顺序、并行加载。

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X, y)          # 把 (样本, 标签) 配对
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0)

print("样本数      :", len(dataset))
print("batch 大小  :", loader.batch_size)
print("一个 epoch 有几个 batch:", len(loader))

for xb, yb in loader:
    print("\n第一批: xb", tuple(xb.shape), " yb", tuple(yb.shape))
    break

### batch / step / epoch 的关系

| 概念 | 意思 |
|---|---|
| **batch** | 一次喂进模型的样本数（32） |
| **step** | 一次参数更新（`optimizer.step()`） |
| **epoch** | 把整个数据集过一遍 |

所以：**一个 epoch 的 step 数 = 样本数 / batch_size**。

想训 1000 步就别数 epoch，直接 `while step < 1000`。

In [ ]:
n_epochs = 2
for epoch in range(n_epochs):
    total = 0.0
    for xb, yb in loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    print(f"epoch {epoch} 平均 loss: {total / len(loader):.4f}")

## 3. 训练集 / 验证集

训练 loss 下降不代表模型变好 —— 它可能只是**背下了训练集**。所以要留一份验证集，只用来看、不用来更新参数。

判读规则：

| 现象 | 含义 |
|---|---|
| 训练降、验证也降 | 好 |
| 训练降、验证不降或上升 | 过拟合 |
| 两个都不降 | 学习率、数据或模型容量有问题 |

In [ ]:
# 划分：前 400 训练，后 100 验证
n_train = 400
X_tr, y_tr = X[:n_train], y[:n_train]
X_va, y_va = X[n_train:], y[n_train:]

train_ld = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True, num_workers=0)


def evaluate(model, Xv, yv):
    "推理：不建图、不更新参数、切到 eval 模式"
    model.eval()
    with torch.no_grad():
        loss = loss_fn(model(Xv), yv)
    model.train()
    return loss.item()


model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

for epoch in range(20):
    for xb, yb in train_ld:
        loss = loss_fn(model(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if epoch % 5 == 0 or epoch == 19:
        tr = evaluate(model, X_tr, y_tr)
        va = evaluate(model, X_va, y_va)
        print(f"epoch {epoch:>2} | train {tr:.4f} | val {va:.4f}")

## 4. 学习率

学习率是最该先调的超参。太小：训不动；太大：loss 来回跳甚至发散。

注意：**loss 跳动不等于训练坏了**。要看趋势和最终指标，不要被单个 step 的数字吓到。

In [ ]:
def run(lr, steps=300):
    torch.manual_seed(0)
    m = nn.Linear(1, 1)
    opt = torch.optim.SGD(m.parameters(), lr=lr)
    history = []
    for _ in range(steps):
        loss = loss_fn(m(X_tr), y_tr)
        opt.zero_grad()
        loss.backward()
        opt.step()
        history.append(loss.item())
    return history, m


for lr in [0.001, 0.05, 0.5]:
    hist, m = run(lr)
    print(f"lr={lr:<6} 起始 {hist[0]:>10.4f}  结束 {hist[-1]:>10.4f}  最低 {min(hist):>10.4f}")

## 5. 练习

In [ ]:
# 练习 1：补完一个 epoch 的训练循环，返回平均 loss
def train_one_epoch(model, loader, optimizer):
    total = 0.0
    for xb, yb in loader:
        # ← 在这里写五步
        pass
    return total / len(loader)


model = nn.Linear(1, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.05)
before = loss_fn(model(X_tr), y_tr).item()
avg = train_one_epoch(model, train_ld, opt)
after = loss_fn(model(X_tr), y_tr).item()

check("练习 1：一个 epoch 后 loss 应下降", avg > 0 and after < before,
      hint="pred = model(xb); loss = loss_fn(pred, yb); optimizer.zero_grad(); loss.backward(); optimizer.step(); total += loss.item()")

In [ ]:
# 练习 2：算出一个 epoch 有多少步
n_samples = 1000
batch_size = 64

n_steps = None                  # ← 改这一行

check("练习 2：应为 16（向上取整）", n_steps == 16,
      hint="1000 / 64 向上取整")

In [ ]:
# 练习 3：写一个评估函数，返回预测的准确率（不做反传）
# 数据：X_eval 的 4 个样本，标签 y_eval
X_eval = torch.tensor([[0.0], [1.0], [2.0], [3.0]])
y_eval = torch.tensor([[1.0], [3.0], [5.0], [7.0]])

m = nn.Linear(1, 1)
m.weight.data.fill_(2.0)
m.bias.data.fill_(1.0)


def accuracy(model, X, y, tol=0.1):
    # ← 改这一行：算预测，和真实值比，误差在 tol 以内的算对
    return 0.0


check("练习 3：准确率应为 1.0",
      accuracy(m, X_eval, y_eval) == 1.0,
      hint="model.eval() + with torch.no_grad() 算 pred，然后 ((pred - y).abs() < tol).float().mean()")

In [ ]:
# 练习 4：下面这段代码 loss 不降，找出问题并修好
def broken_train():
    m = nn.Linear(1, 1)
    opt = torch.optim.SGD(m.parameters(), lr=0.05)
    for _ in range(200):
        loss = loss_fn(m(X_tr), y_tr)
        loss.backward()
        opt.step()
    return loss_fn(m(X_tr), y_tr).item()


bad = broken_train()

check("练习 4：修好后 loss 应小于 1.0", bad < 1.0,
      hint="少了 optimizer.zero_grad()，梯度一直在累加")

In [ ]:
# 练习 5：判断两种学习率下，哪个会发散（返回较大的那个 lr）
lr_a, lr_b = 0.01, 2.0

diverged = None                 # ← 改这一行：填会发散的那个

check("练习 5：应选 2.0", diverged == 2.0,
      hint="学习率过大时 loss 会往上升，2.0 明显过大")

## 收尾

回答这三个：

1. 训练循环的五个步骤是什么？哪两步的先后顺序绝对不能换？
2. `batch`、`step`、`epoch` 分别是什么？1000 个样本、batch=64 时一个 epoch 多少步？
3. 训练 loss 在降、验证 loss 在升，说明什么？该怎么处理？

下一节：**B5 · 综合项目**。